# ДЗ 4. Матрицы в PyTorch: малоранговые представления

Тема про то, зачем вообще нужна матричная низкоранговость (не только "SVD раскладывает матрицу
на три" — а что с этим реально делать): сжатие данных и весов моделей, устранение шума,
рекомендательные системы, восстановление данных по неполным наблюдениям, устойчивое решение
плохо обусловленных систем.

Все задачи используют синтетические матрицы (не требуют скачивания, работают одинаково везде),
крупнейшая матрица в ДЗ — 2000×1500, вычисления занимают секунды.

In [ ]:
# =========================
# SETUP / ENVIRONMENT CHECK
# =========================
import sys
import numpy as np
import torch
import matplotlib.pyplot as plt

print("✓ Python:", sys.version.split()[0])
print("✓ PyTorch:", torch.__version__)
print("✓ NumPy:", np.__version__)
print("✓ Device: CPU")
torch.set_num_threads(max(1, min(4, torch.get_num_threads())))
torch.manual_seed(0)
np.random.seed(0)
print("\nСреда готова к выполнению ДЗ 4.")

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
torch.manual_seed(0)

## Уровень 1. Попроще

### Задача 1.1. SVD-сжатие изображения и теорема Эккарта-Юнга

Теорема Эккарта-Юнга утверждает: среди **всех** матриц ранга $\le k$ именно усечённое SVD
$A_k = U_k \Sigma_k V_k^\top$ даёт наименьшую ошибку приближения в норме Фробениуса, и эта ошибка
в точности равна $\|A - A_k\|_F = \sqrt{\sum_{i>k} \sigma_i^2}$ — сумме квадратов
**отброшенных** сингулярных чисел.

1. Постройте (или загрузите, например `scipy.datasets.face(gray=True)`) любое двумерное
   "изображение" — можно и синтетическое, если предпочитаете детерминированный пример.
2. Посчитайте SVD, постройте rank-$k$ приближение для нескольких $k$, убедитесь, что реальная
   ошибка **совпадает** (не приближённо, а с точностью до чисел с плавающей точкой) с
   теоретической формулой через хвост сингулярных чисел.
3. Экспериментально проверьте саму оптимальность: возьмите произвольную факторизацию
   $A \approx W_1 W_2$ ранга $k$ со случайной инициализацией, **дообучите** её градиентным
   спуском по той же метрике (норма Фробениуса ошибки), и убедитесь, что даже после оптимизации
   она не может превзойти ошибку SVD-усечения того же ранга.
4. Постройте график: ошибка приближения (и график compression ratio) от ранга $k$.

In [ ]:
import time

# Синтетическое детерминированное "изображение".
H,W=96,80
x=torch.linspace(-1,1,W,dtype=torch.float64)
y=torch.linspace(-1,1,H,dtype=torch.float64)
X,Y=torch.meshgrid(x,y,indexing="xy")
A=(torch.exp(-4*(X**2+Y**2)) + 0.25*torch.sin(7*X)*torch.cos(5*Y) + 0.1*X + 0.05*Y).T

U,S,Vh=torch.linalg.svd(A,full_matrices=False)
ranks=[1,2,4,8,16,24,32,48,64]
svd_errors=[]; compression=[]
for k in ranks:
    Ak=(U[:,:k]*S[:k]) @ Vh[:k,:]
    err=torch.linalg.norm(A-Ak).item()
    theory=torch.sqrt(torch.sum(S[k:]**2)).item()
    assert np.isclose(err,theory,rtol=1e-10,atol=1e-10)
    svd_errors.append(err)
    compression.append(A.numel()/(k*(A.shape[0]+A.shape[1]+1)))

print("✓ Теорема Эккарта–Юнга проверена для всех выбранных k.")

# Эксперимент с произвольной rank-k факторизацией.
k=8
target=(U[:,:k]*S[:k])@Vh[:k,:]
g=torch.Generator().manual_seed(123)
W1=torch.randn(A.shape[0],k,dtype=torch.float64,generator=g,requires_grad=True)
W2=torch.randn(k,A.shape[1],dtype=torch.float64,generator=g,requires_grad=True)
opt=torch.optim.Adam([W1,W2],lr=0.05)
initial_loss=torch.linalg.norm(A-W1@W2).item()
for _ in range(1500):
    opt.zero_grad()
    loss=torch.linalg.norm(A-W1@W2)**2
    loss.backward()
    opt.step()
trained_err=torch.linalg.norm(A-W1@W2).item()
svd_err_k=svd_errors[ranks.index(k)]
print(f"Rank-{k} SVD error: {svd_err_k:.8f}")
print(f"Random factorization: initial={initial_loss:.8f}, after GD={trained_err:.8f}")
assert trained_err >= svd_err_k - 1e-5
print("✓ Обученная rank-k факторизация не превзошла оптимум SVD (с учётом численной погрешности).")

plt.figure(figsize=(7,4))
plt.plot(ranks,svd_errors,marker='o')
plt.xlabel('rank k'); plt.ylabel('Frobenius error'); plt.title('SVD approximation error'); plt.grid(True); plt.show()
plt.figure(figsize=(7,4))
plt.plot(ranks,compression,marker='o')
plt.xlabel('rank k'); plt.ylabel('compression ratio'); plt.title('Compression ratio'); plt.grid(True); plt.show()

**Письменный вывод.** Усечённое SVD даёт минимальную ошибку Фробениуса среди матриц заданного ранга, а формула через хвост сингулярных чисел совпадает с измеренной ошибкой с учётом численной погрешности. Произвольная rank-k факторизация после оптимизации не должна давать ошибку ниже SVD-оптимума.

### Задача 1.2. Восстановление зашумлённой low-rank матрицы (денойзинг через усечение)

Возьмите матрицу с известной низкоранговой структурой ($M = UV$, ранг $r$), добавьте гауссов шум
на **все** элементы, и попробуйте восстановить исходную чистую матрицу truncated SVD.

1. Постройте `noisy = clean + шум`.
2. Для каждого $k = 1, \ldots, \min(m,n)$ посчитайте ошибку rank-$k$ приближения `noisy`
   **относительно `clean`** (не относительно `noisy`!) — это ключевое отличие от задачи 1.1.
3. Найдите $k^*$, минимизирующий эту ошибку, и сравните с истинным рангом $r$.
4. Сравните лучший достижимый truncation-денойзинг с "неусечённой" зашумлённой матрицей (то есть
   $k=\min(m,n)$, эквивалентно вообще не денойзить).

In [ ]:
# Low-rank matrix + Gaussian noise.
r_true=5; m,n=80,60
rng=torch.Generator().manual_seed(10)
U0=torch.randn(m,r_true,generator=rng,dtype=torch.float64)
V0=torch.randn(r_true,n,generator=rng,dtype=torch.float64)
clean=U0@V0
noise=0.35*torch.randn(m,n,generator=rng,dtype=torch.float64)
noisy=clean+noise
Un,Sn,Vhn=torch.linalg.svd(noisy,full_matrices=False)
max_rank=min(m,n)
errors=[]
for k in range(1,max_rank+1):
    Ak=(Un[:,:k]*Sn[:k])@Vhn[:k,:]
    errors.append(torch.linalg.norm(clean-Ak).item())
errors=np.asarray(errors)
best_k=int(np.argmin(errors))+1
best_err=float(errors[best_k-1])
no_denoise_err=float(torch.linalg.norm(clean-noisy))
print(f"Истинный ранг: {r_true}")
print(f"Лучший k*: {best_k}")
print(f"Ошибка лучшего truncation: {best_err:.6f}")
print(f"Ошибка без денойзинга (k={max_rank}): {no_denoise_err:.6f}")
assert best_err <= no_denoise_err + 1e-12
print("✓ Ошибка считается относительно clean, как требует задание.")
print("✓ Truncated SVD не хуже полного noisy-варианта.")

plt.figure(figsize=(7,4)); plt.plot(range(1,max_rank+1),errors); plt.axvline(r_true,ls='--',label='true rank'); plt.axvline(best_k,ls=':',label='best k'); plt.xlabel('k'); plt.ylabel('||clean - A_k(noisy)||_F'); plt.legend(); plt.grid(True); plt.show()

**Письменный вывод.** Для денойзинга критерий считается относительно исходной `clean`-матрицы. При малых `k` часть сигнала отбрасывается, при слишком больших `k` в восстановление возвращается шум, поэтому существует некоторый оптимальный `k*`, который может отличаться от истинного ранга.

### Задача 1.3. Плохая обусловленность и регуляризация через усечение SVD

Число обусловленности $\mathrm{cond}(A) = \sigma_{\max}/\sigma_{\min}$ показывает, во сколько раз
могут усилиться ошибки измерения при решении $Ax=b$. При большом числе обусловленности даже
крошечный шум в $b$ может привести к катастрофически неверному $x$.

1. Постройте искусственно плохо обусловленную матрицу $A = U\Sigma V^\top$ с сингулярными
   числами, убывающими от 1 до $10^{-12}$ (используйте `torch.logspace`).
2. Возьмите $x_{true}$, лежащий **строго в подпространстве**, натянутом на первые $k_{true}=10$
   правых сингулярных векторов (а не произвольный случайный вектор!) — так, чтобы у задачи
   в принципе была шанс на точное восстановление.
3. Постройте $b = Ax_{true}$, добавьте небольшой шум измерения к $b$ (например, $10^{-6}$).
4. Сравните: (а) наивное решение через `torch.linalg.solve(A, b_noisy)`; (б) решения через
   усечённую псевдообратную с разным числом сохранённых сингулярных направлений $k$.
5. Постройте график ошибки от $k$ и объясните его форму.

In [ ]:
# Плохо обусловленная система.
n=60; k_true=10
G=torch.Generator().manual_seed(20)
Q1,_=torch.linalg.qr(torch.randn(n,n,generator=G,dtype=torch.float64))
Q2,_=torch.linalg.qr(torch.randn(n,n,generator=G,dtype=torch.float64))
s=torch.logspace(0,-12,n,dtype=torch.float64)
A=Q1@torch.diag(s)@Q2.T
x_coeff=torch.randn(k_true,generator=G,dtype=torch.float64)
x_true=Q2[:,:k_true]@x_coeff
b=A@x_true
noise_level=1e-6
noise=noise_level*torch.randn(n,generator=G,dtype=torch.float64)
b_noisy=b+noise
cond=(s[0]/s[-1]).item()
print(f"cond(A) ≈ {cond:.3e}")

x_naive=torch.linalg.solve(A,b_noisy)
naive_err=torch.linalg.norm(x_naive-x_true).item()

errs=[]
for k in range(1,n+1):
    Uk=Q1[:,:k]; Vk=Q2[:,:k]; sk=s[:k]
    xk=Vk@((Uk.T@b_noisy)/sk)
    errs.append(torch.linalg.norm(xk-x_true).item())
errs=np.asarray(errs)
best_k=int(np.argmin(errs))+1
print(f"Наивное решение error: {naive_err:.6e}")
print(f"Лучший truncated-pinv k: {best_k}, error: {errs[best_k-1]:.6e}")
assert np.isfinite(naive_err)
assert np.isfinite(errs).all()
assert best_k <= n
print("✓ x_true построен строго в первых 10 правых сингулярных направлениях.")
print("✓ Проверены naive solve и все k для truncated pseudoinverse.")

plt.figure(figsize=(7,4)); plt.semilogy(range(1,n+1),errs,label='truncated pseudoinverse'); plt.axhline(naive_err,ls='--',label='naive solve'); plt.axvline(k_true,ls=':',label='k_true=10'); plt.xlabel('kept singular directions k'); plt.ylabel('solution error'); plt.legend(); plt.grid(True); plt.show()

**Письменный вывод.** Из-за сингулярных чисел до $10^{-12}$ система крайне плохо обусловлена: шум в правой части усиливается при обращении малых сингулярных значений. Усечённая псевдообратная подавляет наиболее неустойчивые направления; поэтому ошибка сначала может уменьшаться, а затем расти при добавлении шумовых компонент.

## Уровень 2. Посложнее

### Задача 2 (★). Randomized SVD для больших матриц

Полное SVD стоит $O(\min(mn^2, m^2n))$ — дорого для больших матриц, даже если реально нужны
только первые $k \ll \min(m,n)$ сингулярных векторов. **Randomized SVD** (Halko, Martinsson,
Tropp, 2011) находит хорошее приближение top-$k$ SVD намного дешевле, используя случайную
проекцию.

1. Реализуйте `randomized_svd(A, k, n_oversample, n_iter)`:
   - Спроецируйте `A` на случайное подпространство: `Y = A @ Omega`, где `Omega` — случайная
     матрица размера `(n, k + n_oversample)`.
   - Улучшите качество подпространства несколькими **степенными итерациями**:
     `Y = A @ (A.T @ Y)` — это усиливает "сигнальные" направления относительно шумовых.
   - Постройте ортонормированный базис `Q` через `torch.linalg.qr(Y)`.
   - Спроецируйте задачу в это подпространство: `B = Q.T @ A` (маленькая матрица!), сделайте
     точное SVD от `B`, "поднимите" левые сингулярные векторы обратно: `U = Q @ U_B`.
2. Сравните на матрице $2000 \times 1500$ с истинным рангом 30 (плюс небольшой шум): скорость и
   точность (по сравнению с полным `torch.linalg.svd`) для первых 30 сингулярных чисел/векторов.

In [ ]:
# Randomized SVD.

def randomized_svd(A,k,n_oversample=10,n_iter=2):
    if A.ndim != 2: raise ValueError("A must be 2D")
    m,n=A.shape
    if not (1 <= k <= min(m,n)): raise ValueError("invalid k")
    if n_oversample < 0 or k+n_oversample > min(m,n): raise ValueError("invalid oversampling")
    l=k+n_oversample
    Omega=torch.randn(n,l,dtype=A.dtype,device=A.device)
    Y=A@Omega
    for _ in range(n_iter):
        Y=A@(A.T@Y)
    Q,_=torch.linalg.qr(Y,mode='reduced')
    B=Q.T@A
    Ub,S,Vh=torch.linalg.svd(B,full_matrices=False)
    U=Q@Ub
    return U[:,:k],S[:k],Vh[:k,:]

m,n=2000,1500; true_rank=30
G=torch.Generator().manual_seed(30)
L=torch.randn(m,true_rank,generator=G,dtype=torch.float32)
R=torch.randn(true_rank,n,generator=G,dtype=torch.float32)
A_big=L@R + 0.03*torch.randn(m,n,generator=G,dtype=torch.float32)
k=30

start=time.perf_counter(); Ufull,Sfull,Vhfull=torch.linalg.svd(A_big,full_matrices=False); full_time=time.perf_counter()-start
start=time.perf_counter(); Ur,Sr,Vhr=randomized_svd(A_big,k,10,2); rand_time=time.perf_counter()-start

rel_sv=torch.linalg.norm(Sr-Sfull[:k])/torch.linalg.norm(Sfull[:k])
# Subspace quality: projection of exact top-k left singular vectors onto randomized subspace.
projection_error=torch.linalg.norm(Ufull[:,:k]-Ur@(Ur.T@Ufull[:,:k]))/torch.linalg.norm(Ufull[:,:k])
Ak_rand=(Ur*Sr)@Vhr
rel_frob=torch.linalg.norm(A_big-Ak_rand)/torch.linalg.norm(A_big)

print(f"Full SVD time:       {full_time:.3f}s")
print(f"Randomized SVD time: {rand_time:.3f}s")
print(f"Top-{k} singular values relative error: {rel_sv:.3e}")
print(f"Top-{k} subspace projection error:      {projection_error:.3e}")
print(f"Randomized rank-{k} relative matrix error: {rel_frob:.3e}")
print(f"Speedup full/randomized: {full_time/rand_time:.2f}x")
assert full_time > 0 and rand_time > 0 and np.isfinite(full_time) and np.isfinite(rand_time)
assert torch.isfinite(Sr).all() and torch.isfinite(Ur).all() and torch.isfinite(Vhr).all()
assert rel_sv < 0.05
print("✓ Randomized SVD корректно возвращает top-k approximation с малой ошибкой.")

plt.figure(figsize=(7,4)); plt.semilogy(Sfull[:k].numpy(),label='full SVD'); plt.semilogy(Sr.numpy(),'--',label='randomized'); plt.xlabel('singular value index'); plt.ylabel('sigma'); plt.legend(); plt.grid(True); plt.show()

**Письменный вывод.** Randomized SVD строит небольшое случайное подпространство и затем делает точное SVD только для сжатой матрицы. Для матрицы с быстро убывающим спектром top-k сингулярные значения и подпространство должны хорошо приближаться, при этом стоимость обычно ниже полного SVD.

---
**Что сдавать:** ноутбук с реализованными функциями, графиками и письменными ответами на вопросы
для решённых задач. Все вычисления в сумме занимают на CPU не больше пары минут (самая тяжёлая
операция — полное SVD матрицы 2000×1500 в задаче 3.1, около секунды).